In [11]:
import os

print("Notebook is running from:")
print(os.getcwd())

Notebook is running from:
c:\Data Science\Project\Analysis\Citi Bike\citibike-project\notebooks


In [12]:
import os

print("Folders/files here:")
print(os.listdir())

Folders/files here:
['01_data_collection.ipynb', '02_data_cleaning.ipynb', '03_eda.ipynb', '04_linear_regression.ipynb']


In [13]:
import os

print("Does data folder exist?", os.path.exists("data"))
print("Does data/raw folder exist?", os.path.exists("data/raw"))

Does data folder exist? False
Does data/raw folder exist? False


In [14]:
import glob

files = glob.glob("../data/raw/*")

print("Files found:")
for file in files:
    print(file)

Files found:
../data/raw\202608-citibike-tripdata_1.csv
../data/raw\202608-citibike-tripdata_2.csv
../data/raw\202608-citibike-tripdata_3.csv
../data/raw\202608-citibike-tripdata_4.csv
../data/raw\202608-citibike-tripdata_5.csv
../data/raw\202608-citibike-tripdata_6.csv


STEP 1: Load All Datasets

In [21]:
import pandas as pd
import numpy as np
import glob

# 1. Load all Citi Bike files (if multiple files for August)
files = glob.glob("../data/raw/*citibike*.csv")
trips_list = [pd.read_csv(f) for f in files]
trips = pd.concat(trips_list, ignore_index=True)

# 2. Load Weather
weather = pd.read_csv("../data/external/nyc_weather.csv")

# 3. Load Station Information
stations = pd.read_csv("../data/external/station_information.csv")

print("Trips shape:", trips.shape)
print("Weather shape:", weather.shape)
print("Stations shape:", stations.shape)

C:\Users\Ritesh\AppData\Local\Temp\ipykernel_19712\1986093131.py:7: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  trips_list = [pd.read_csv(f) for f in files]


Trips shape: (5246236, 13)
Weather shape: (2208, 5)
Stations shape: (2520, 15)


STEP 2: Clean Trip Data

In [22]:
# Convert to datetime
trips['started_at'] = pd.to_datetime(trips['started_at'])

trips['ended_at'] = pd.to_datetime(trips['ended_at'])

# Calculate trip duration in minutes
trips['trip_duration_min'] = (trips['ended_at'] - trips['started_at']).dt.total_seconds() / 60

# Remove invalid trips
trips = trips[(trips['trip_duration_min'] > 1) & (trips['trip_duration_min'] < 24*60)]

# Remove duplicates
trips = trips.drop_duplicates(subset=['ride_id'])

print("After cleaning trips:", trips.shape)

After cleaning trips: (5244973, 14)


STEP 3: Clean Station Data

In [23]:
# Keep only useful columns
stations = stations[['station_id', 'name', 'lat', 'lon', 'capacity']].copy()

# Rename columns
stations = stations.rename(columns={
'name': 'station_name',
'lat': 'latitude',
'lon': 'longitude'
})

STEP 4: Merge Trips + Station Information

In [24]:
# ----- Start Station Merge -----

trips = trips.merge(
stations,
left_on='start_station_id',
right_on='station_id',
how='left'
)

trips = trips.rename(columns={
'station_name': 'start_station_name_clean',
'latitude': 'start_lat_station',
'longitude': 'start_lng_station',
'capacity': 'start_capacity'
})
trips = trips.drop(columns=['station_id'])

# ----- End Station Merge -----
trips = trips.merge(
stations,
left_on='end_station_id',
right_on='station_id',
how='left'
)

trips = trips.rename(columns={
'station_name': 'end_station_name_clean',
'latitude': 'end_lat_station',
'longitude': 'end_lng_station',
'capacity': 'end_capacity'

})
trips = trips.drop(columns=['station_id'])

STEP 5: Merge with Weather Data

In [26]:
# Prepare weather
weather['time'] = pd.to_datetime(weather['time'])
weather['date_hour'] = weather['time'].dt.floor('h')

# Prepare trips
trips['date_hour'] = trips['started_at'].dt.floor('h')

# Merge
final_df = trips.merge(weather, on='date_hour', how='left')

STEP 6: Feature Engineering

In [29]:
import numpy as np

# Time features
final_df['hour'] = final_df['started_at'].dt.hour
final_df['day_of_week'] = final_df['started_at'].dt.day_name()
final_df['is_weekend'] = final_df['started_at'].dt.dayofweek >= 5

# Trip Distance (Haversine)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

final_df['trip_distance_km'] = haversine(
    final_df['start_lat'], final_df['start_lng'],
    final_df['end_lat'], final_df['end_lng']
)

STEP 7: Select Final Columns & Save

In [33]:
final_columns = [
'ride_id', 'rideable_type', 'started_at', 'ended_at',
'trip_duration_min', 'trip_distance_km', 'member_casual',

'start_station_id', 'start_station_name', 'start_lat', 'start_lng', 'start_capacity',
'end_station_id', 'end_station_name', 'end_lat', 'end_lng', 'end_capacity',

'hour', 'day_of_week', 'is_weekend',

'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m'
]

final_df = final_df[final_columns]

# Save cleaned data
final_df.to_csv("../data/processed/citibike_final_cleaned.csv", index=False)
# or
#final_df.to_parquet("../data/processed/citibike_final_cleaned.parquet")

print("Final shape:", final_df.shape)
print(final_df.head())

Final shape: (5244973, 24)
            ride_id  rideable_type              started_at  \
0  E4A8132DACF670C3  electric_bike 2026-08-10 14:31:22.100   
1  8382768C05C30BB4  electric_bike 2026-08-10 08:04:28.348   
2  3A1C43832BBE4FCD   classic_bike 2026-08-08 19:36:32.350   
3  5C9BC6DFC4DEA183  electric_bike 2026-08-05 14:34:27.749   
4  355579CC64D1BE20  electric_bike 2026-08-01 14:16:44.531   

                 ended_at  trip_duration_min  trip_distance_km member_casual  \
0 2026-08-10 14:37:37.034           6.248900          0.928516        member   
1 2026-08-10 08:33:27.680          28.988867          6.342994        member   
2 2026-08-08 19:43:47.890           7.259000          0.855081        member   
3 2026-08-05 14:46:23.526          11.929617          2.271952        member   
4 2026-08-01 14:29:39.539          12.916800          2.271952        member   

  start_station_id         start_station_name  start_lat  ...    end_lat  \
0          6322.01            E 33 St & 5 A

In [34]:
final_df

,ride_id,rideable_type,started_at,ended_at,trip_duration_min,trip_distance_km,member_casual,start_station_id,start_station_name,start_lat,...,end_lat,end_lng,end_capacity,hour,day_of_week,is_weekend,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,E4A8132DACF670C3,electric_bike,2026-08-10 14:31:22.100,2026-08-10 14:37:37.034,6.248900,0.928516,member,6322.01,E 33 St & 5 Ave,40.747659,...,40.744876,-73.995299,NaN,14,Monday,False,NaN,NaN,NaN,NaN
1,8382768C05C30BB4,electric_bike,2026-08-10 08:04:28.348,2026-08-10 08:33:27.680,28.988867,6.342994,member,4683.02,Emerson Pl & Myrtle Ave,40.693631,...,40.744876,-73.995299,NaN,8,Monday,False,NaN,NaN,NaN,NaN
2,3A1C43832BBE4FCD,classic_bike,2026-08-08 19:36:32.350,2026-08-08 19:43:47.890,7.259000,0.855081,member,5980.11,Broadway & E 19 St,40.738290,...,40.744876,-73.995299,NaN,19,Saturday,True,NaN,NaN,NaN,NaN
3,5C9BC6DFC4DEA183,electric_bike,2026-08-05 14:34:27.749,2026-08-05 14:46:23.526,11.929617,2.271952,member,3993.03,Sterling Pl & Bedford Ave,40.672695,...,40.687534,-73.972652,NaN,14,Wednesday,False,NaN,NaN,NaN,NaN
4,355579CC64D1BE20,electric_bike,2026-08-01 14:16:44.531,2026-08-01 14:29:39.539,12.916800,2.271952,member,3993.03,Sterling Pl & Bedford Ave,40.672695,...,40.687534,-73.972652,NaN,14,Saturday,True,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5244968,7778C182C86ED253,electric_bike,2026-08-29 16:40:59.109,2026-08-29 16:52:10.441,11.188867,1.601043,member,3303.03,Caton Ave & Argyle Rd,40.649681,...,40.661063,-73.979453,NaN,16,Saturday,True,NaN,NaN,NaN,NaN
5244969,7E1978AE88707549,electric_bike,2026-08-30 14:40:57.513,2026-08-30 14:45:19.773,4.371000,1.027000,member,6089.08,Lexington Ave & E 26 St,40.741459,...,40.749499,-73.977292,NaN,14,Sunday,True,NaN,NaN,NaN,NaN
5244970,EE79B9E27476A26B,electric_bike,2026-08-29 17:44:45.322,2026-08-29 18:18:35.303,33.833017,6.762998,member,5489.03,Kent Ave & N 7 St,40.720368,...,40.661063,-73.979453,NaN,17,Saturday,True,NaN,NaN,NaN,NaN
5244971,2C83FD498727AFDD,electric_bike,2026-08-31 17:26:01.392,2026-08-31 17:36:15.940,10.242467,1.827469,casual,6239.08,E 31 St & 3 Ave,40.743943,...,40.756458,-73.993722,NaN,17,Monday,False,NaN,NaN,NaN,NaN
